# GraphRAG Pipeline for AI Company Knowledge Graphs

## Project Goal

This notebook implements a complete **GraphRAG (Graph Retrieval-Augmented Generation)** pipeline applied to a knowledge base of major AI companies.

### What is GraphRAG?
Traditional RAG systems retrieve flat text chunks based on semantic similarity. GraphRAG enhances this by:
1. **Structuring knowledge** as a graph of entities and relationships
2. **Traversing connections** between entities via multi-hop reasoning
3. **Providing richer context** that captures how entities relate to each other

### Pipeline Overview
1. Fetch Wikipedia articles for 20 major AI companies
2. Extract (subject, predicate, object) triples using an LLM
3. Build a NetworkX knowledge graph with embeddings on each node
4. Visualize the graph with PyVis
5. Implement Flat RAG (FAISS-based) as a baseline
6. Implement GraphRAG with BFS-based subgraph retrieval
7. Benchmark both systems on multi-hop reasoning questions
8. Evaluate, compare, and analyze costs

### Why AI Companies?
The AI industry has rich interconnections — shared founders, cross-investments, talent flows, and partnerships — making it an ideal domain for testing multi-hop reasoning capabilities.

## Cell 1: Setup & Imports

Import all necessary libraries for the pipeline.

In [ ]:
import os, json, time, warnings
warnings.filterwarnings('ignore')
import networkx as nx
import numpy as np
import pandas as pd
from openai import OpenAI
import faiss
import requests
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pyvis.network import Network
from IPython.display import display, HTML, IFrame
import pickle
from collections import deque

print("All imports successful!")

## Cell 2: Configuration

Set up the OpenAI client and define model constants.

In [ ]:
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
MODEL = "gpt-4o-mini"
EMBED_MODEL = "text-embedding-3-small"
EMBED_DIM = 1536

print(f"Model: {MODEL}")
print(f"Embedding Model: {EMBED_MODEL}")
print(f"Embedding Dimensions: {EMBED_DIM}")

## Cell 3: Fetch Wikipedia Articles

We fetch Wikipedia articles for 20 major AI companies using the Wikipedia REST API.
Content is limited to 3000 characters per article to manage token costs.

In [ ]:
%%time

COMPANIES = [
    "OpenAI", "Google DeepMind", "Anthropic", "Mistral AI", "Inflection AI",
    "Stability AI", "Cohere", "Hugging Face", "Runway (company)", "Scale AI",
    "Cerebras Systems", "Together AI", "Perplexity AI", "Character.AI",
    "xAI (company)", "Adept (company)", "Aleph Alpha", "01.AI", "Moonshot AI",
    "Baidu"
]

def fetch_wikipedia_article(title, max_chars=3000):
    """Fetch a Wikipedia article using the MediaWiki API."""
    try:
        # Use Wikipedia's REST API to get the article extract
        url = "https://en.wikipedia.org/w/api.php"
        params = {
            "action": "query",
            "format": "json",
            "titles": title,
            "prop": "extracts",
            "exintro": False,
            "explaintext": True,
            "exsectionformat": "plain",
            "redirects": 1
        }
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()
        
        pages = data.get("query", {}).get("pages", {})
        for page_id, page in pages.items():
            if page_id == "-1":
                return None
            content = page.get("extract", "")
            if content:
                return content[:max_chars]
        return None
    except Exception as e:
        print(f"  Error fetching '{title}': {e}")
        return None

articles = []
for i, company in enumerate(COMPANIES):
    print(f"[{i+1}/{len(COMPANIES)}] Fetching: {company}")
    content = fetch_wikipedia_article(company)
    if content:
        articles.append({"title": company, "content": content})
        print(f"  -> Got {len(content)} chars")
    else:
        print(f"  -> Not found, skipping")
    time.sleep(0.2)  # Be polite to Wikipedia

print(f"\nSuccessfully fetched {len(articles)} articles")

## Cell 4: Entity Extraction — Building the Triple Store

For each article, we use GPT-4o-mini to extract (subject, predicate, object) triples.
These form the edges and nodes of our knowledge graph.

Examples of triples:
- `(OpenAI, founded_by, Sam Altman)`
- `(Microsoft, invested_in, OpenAI)`
- `(Anthropic, located_in, San Francisco)`

In [ ]:
%%time

EXTRACTION_SYSTEM_PROMPT = """You are a knowledge graph extractor. Extract (subject, predicate, object) triples from the text about AI companies.
Focus on: founders, employees, investments, products, acquisitions, partnerships, locations.
Return JSON: {"triples": [{"subject": "...", "predicate": "...", "object": "..."}]}
Keep subject/object as proper nouns. Max 20 triples per text."""

def extract_triples(text, title):
    """Extract knowledge graph triples from article text."""
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": EXTRACTION_SYSTEM_PROMPT},
                {"role": "user", "content": f"Extract triples from this article about {title}:\n\n{text}"}
            ],
            response_format={"type": "json_object"},
            temperature=0
        )
        result = json.loads(response.choices[0].message.content)
        return result.get("triples", [])
    except Exception as e:
        print(f"  Error extracting triples for '{title}': {e}")
        return []

all_triples = []
token_counts = {"extraction": 0}

for i, article in enumerate(articles):
    print(f"[{i+1}/{len(articles)}] Extracting triples: {article['title']}")
    triples = extract_triples(article["content"], article["title"])
    all_triples.extend(triples)
    print(f"  -> Extracted {len(triples)} triples (total: {len(all_triples)})")
    time.sleep(0.5)  # Rate limiting

# Save to disk
with open("/Users/jin/Day19/triples.json", "w") as f:
    json.dump(all_triples, f, indent=2)

print(f"\nTotal triples extracted: {len(all_triples)}")
print(f"Saved to triples.json")
print("\nSample triples:")
for t in all_triples[:5]:
    print(f"  ({t['subject']}, {t['predicate']}, {t['object']})")

## Cell 5: Build NetworkX Graph

Convert the extracted triples into a directed NetworkX graph.
Each triple `(S, P, O)` becomes:
- Two nodes: S and O
- One directed edge S → O with label P

In [ ]:
G = nx.DiGraph()

for triple in all_triples:
    try:
        s = triple["subject"]
        p = triple["predicate"]
        o = triple["object"]
        G.add_node(s)
        G.add_node(o)
        G.add_edge(s, o, relation=p)
    except KeyError:
        continue  # Skip malformed triples

print(f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")
print(f"\nTop 10 nodes by degree (in+out):")
degree_sorted = sorted(G.degree(), key=lambda x: x[1], reverse=True)[:10]
for node, deg in degree_sorted:
    print(f"  {node}: {deg}")

## Cell 6: Add Node Embeddings

For each node (entity) in the graph, compute a semantic embedding using `text-embedding-3-small`.
These embeddings enable:
- Fuzzy matching (find nodes semantically similar to a query)
- Seed node identification in GraphRAG

We process in batches of 50 for efficiency.

In [ ]:
%%time

def get_embedding(text):
    """Get embedding for a single text string."""
    try:
        response = client.embeddings.create(
            model=EMBED_MODEL,
            input=text
        )
        return response.data[0].embedding
    except Exception as e:
        print(f"  Error getting embedding for '{text}': {e}")
        return [0.0] * EMBED_DIM

def get_embeddings_batch(texts, batch_size=50):
    """Get embeddings for a list of texts in batches."""
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        try:
            response = client.embeddings.create(
                model=EMBED_MODEL,
                input=batch
            )
            batch_embeddings = [item.embedding for item in response.data]
            all_embeddings.extend(batch_embeddings)
            print(f"  Embedded batch {i//batch_size + 1}/{(len(texts)-1)//batch_size + 1} ({len(all_embeddings)}/{len(texts)} nodes)")
        except Exception as e:
            print(f"  Error in batch {i//batch_size + 1}: {e}")
            all_embeddings.extend([[0.0] * EMBED_DIM] * len(batch))
        time.sleep(0.1)
    return all_embeddings

# Get all node names and compute embeddings in batches
nodes = list(G.nodes())
print(f"Computing embeddings for {len(nodes)} nodes...")
node_embeddings = get_embeddings_batch(nodes, batch_size=50)

# Store embeddings in graph
for node, embedding in zip(nodes, node_embeddings):
    G.nodes[node]["embedding"] = embedding

# Save graph
with open("/Users/jin/Day19/graph.pkl", "wb") as f:
    pickle.dump(G, f)

print(f"\nEmbeddings computed for {len(nodes)} nodes")
print(f"Graph saved to graph.pkl")

## Cell 7: Visualize the Knowledge Graph

We create two visualizations:
1. **Interactive PyVis graph** (`knowledge_graph.html`) — explore connections in the browser
2. **Static matplotlib bar chart** — top 20 nodes by degree centrality

Color coding:
- **Blue**: AI company nodes (appear in our article titles)
- **Orange**: People, concepts, and other entities

In [ ]:
# Identify company nodes
company_titles = set()
for a in articles:
    # Add both the original title and cleaned version
    title = a["title"]
    company_titles.add(title)
    # Also add cleaned versions without parentheses
    clean = title.split(" (")[0]
    company_titles.add(clean)

# --- PyVis Interactive Graph ---
net = Network(
    height="700px",
    width="100%",
    bgcolor="#1a1a2e",
    font_color="white",
    directed=True
)
net.barnes_hut(gravity=-8000, central_gravity=0.3, spring_length=150)

# Only include nodes with degree >= 2 to reduce clutter
min_degree = 2
filtered_nodes = [n for n in G.nodes() if G.degree(n) >= min_degree]
filtered_graph = G.subgraph(filtered_nodes)

print(f"Showing {len(filtered_nodes)} nodes with degree >= {min_degree} (out of {G.number_of_nodes()} total)")

for node in filtered_graph.nodes():
    is_company = any(c.lower() in node.lower() or node.lower() in c.lower() for c in company_titles)
    color = "#4a90d9" if is_company else "#e8a838"  # Blue for companies, orange for others
    size = max(10, min(40, filtered_graph.degree(node) * 3))
    net.add_node(
        node,
        label=node,
        color=color,
        size=size,
        title=f"{node}\nDegree: {filtered_graph.degree(node)}"
    )

for s, o, data in filtered_graph.edges(data=True):
    relation = data.get("relation", "")
    net.add_edge(s, o, title=relation, label=relation[:20], color="#888888", arrows="to")

# Save
graph_html_path = "/Users/jin/Day19/knowledge_graph.html"
net.save_graph(graph_html_path)
print(f"Interactive graph saved to: {graph_html_path}")
print("Open in browser to explore interactively.")

# Display link
display(HTML(f'<a href="{graph_html_path}" target="_blank">Open Interactive Knowledge Graph</a>'))

In [ ]:
# --- Matplotlib: Top 20 Nodes by Degree ---
top_nodes = sorted(G.degree(), key=lambda x: x[1], reverse=True)[:20]
names = [n[0][:30] for n in top_nodes]  # Truncate long names
degrees = [n[1] for n in top_nodes]

colors = []
for name in [n[0] for n in top_nodes]:
    is_company = any(c.lower() in name.lower() or name.lower() in c.lower() for c in company_titles)
    colors.append("#4a90d9" if is_company else "#e8a838")

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(range(len(names)), degrees, color=colors)
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=10)
ax.set_xlabel("Degree (in + out)", fontsize=12)
ax.set_title("Top 20 Nodes by Degree in AI Knowledge Graph", fontsize=14, fontweight="bold")

# Legend
company_patch = mpatches.Patch(color="#4a90d9", label="AI Company")
other_patch = mpatches.Patch(color="#e8a838", label="Person/Concept")
ax.legend(handles=[company_patch, other_patch], loc="lower right")

ax.invert_yaxis()
plt.tight_layout()
plt.savefig("/Users/jin/Day19/top_nodes.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved to top_nodes.png")

## Cell 8: Flat RAG Implementation (Baseline)

Traditional RAG pipeline:
1. **Chunk** articles into 500-char segments with 50-char overlap
2. **Embed** all chunks with `text-embedding-3-small`
3. **Index** with FAISS for fast similarity search
4. **Retrieve** top-k chunks at query time
5. **Generate** answer using LLM with retrieved context

This is our baseline — no graph traversal, just semantic similarity.

In [ ]:
%%time

def chunk_text(text, chunk_size=500, overlap=50):
    """Split text into overlapping chunks."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

# Build corpus of chunks
all_chunks = []
chunk_metadata = []  # Track which article each chunk came from

for article in articles:
    chunks = chunk_text(article["content"])
    for i, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        chunk_metadata.append({"title": article["title"], "chunk_idx": i})

print(f"Total chunks: {len(all_chunks)}")
print(f"Average chunk size: {sum(len(c) for c in all_chunks) / len(all_chunks):.0f} chars")

# Embed all chunks
print("\nEmbedding chunks...")
chunk_embeddings = get_embeddings_batch(all_chunks, batch_size=50)
chunk_matrix = np.array(chunk_embeddings, dtype="float32")

# Build FAISS index
faiss.normalize_L2(chunk_matrix)
index = faiss.IndexFlatIP(EMBED_DIM)  # Inner product (cosine sim after normalization)
index.add(chunk_matrix)
print(f"\nFAISS index built with {index.ntotal} vectors")

In [ ]:
def flat_rag_query(question, k=5):
    """Answer a question using flat RAG (FAISS retrieval + LLM generation)."""
    # Embed question
    q_embedding = np.array([get_embedding(question)], dtype="float32")
    faiss.normalize_L2(q_embedding)
    
    # Retrieve top-k chunks
    scores, indices = index.search(q_embedding, k)
    
    # Build context
    context_parts = []
    for score, idx in zip(scores[0], indices[0]):
        if idx >= 0:
            meta = chunk_metadata[idx]
            context_parts.append(f"[Source: {meta['title']}]\n{all_chunks[idx]}")
    
    context = "\n\n---\n\n".join(context_parts)
    
    # Generate answer
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful assistant. Answer the question based on the provided context. Be specific and cite companies/people mentioned."},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
        ],
        temperature=0
    )
    
    return response.choices[0].message.content

# Quick test
print("Testing Flat RAG...")
test_answer = flat_rag_query("Who founded OpenAI?")
print(f"Q: Who founded OpenAI?")
print(f"A: {test_answer[:200]}...")

## Cell 9: GraphRAG Implementation

GraphRAG leverages the knowledge graph structure for retrieval:

1. **Entity extraction**: Identify key entities in the question
2. **Seed node matching**: Find those entities in the graph (exact + fuzzy)
3. **BFS traversal**: Expand the subgraph up to `max_hops` from seed nodes
4. **Subgraph textualization**: Convert graph structure to readable text
5. **LLM generation**: Answer using the rich relational context

This enables **multi-hop reasoning**: e.g., "Dario Amodei → worked at → OpenAI → therefore → Anthropic founders previously at OpenAI"

In [ ]:
def extract_entities_from_question(question):
    """Use LLM to extract key entities from a question."""
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": "Extract key named entities (companies, people, organizations) from the question. Return JSON: {\"entities\": [...]}"},
                {"role": "user", "content": question}
            ],
            response_format={"type": "json_object"},
            temperature=0
        )
        result = json.loads(response.choices[0].message.content)
        return result.get("entities", [])
    except Exception as e:
        print(f"Entity extraction error: {e}")
        return []

def find_seed_nodes(entities, graph, top_k=3):
    """Find matching nodes in the graph for given entities.
    Uses exact match first, then embedding-based fuzzy match."""
    seed_nodes = set()
    graph_nodes = list(graph.nodes())
    
    for entity in entities:
        entity_lower = entity.lower()
        
        # 1. Exact match
        for node in graph_nodes:
            if node.lower() == entity_lower:
                seed_nodes.add(node)
                break
        
        # 2. Substring match
        if entity not in seed_nodes:
            for node in graph_nodes:
                if entity_lower in node.lower() or node.lower() in entity_lower:
                    seed_nodes.add(node)
        
        # 3. Embedding-based fuzzy match (if still not found)
        if not seed_nodes or entity not in str(seed_nodes):
            try:
                entity_emb = np.array(get_embedding(entity), dtype="float32")
                best_score = -1
                best_node = None
                for node in graph_nodes:
                    node_emb = graph.nodes[node].get("embedding")
                    if node_emb:
                        node_arr = np.array(node_emb, dtype="float32")
                        score = float(np.dot(entity_emb, node_arr) / 
                                     (np.linalg.norm(entity_emb) * np.linalg.norm(node_arr) + 1e-8))
                        if score > best_score:
                            best_score = score
                            best_node = node
                if best_node and best_score > 0.7:
                    seed_nodes.add(best_node)
            except Exception:
                pass
    
    return list(seed_nodes)[:top_k]

def bfs_subgraph(graph, seed_nodes, max_hops=2):
    """BFS from seed nodes to collect a local subgraph."""
    visited = set(seed_nodes)
    queue = deque([(node, 0) for node in seed_nodes])
    subgraph_edges = []
    
    while queue:
        node, hop = queue.popleft()
        if hop >= max_hops:
            continue
        
        # Traverse outgoing edges
        for neighbor in graph.successors(node):
            edge_data = graph[node][neighbor]
            subgraph_edges.append((node, edge_data.get("relation", "related_to"), neighbor))
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append((neighbor, hop + 1))
        
        # Also traverse incoming edges for richer context
        for predecessor in graph.predecessors(node):
            edge_data = graph[predecessor][node]
            subgraph_edges.append((predecessor, edge_data.get("relation", "related_to"), node))
            if predecessor not in visited:
                visited.add(predecessor)
                queue.append((predecessor, hop + 1))
    
    return visited, subgraph_edges

def textualize_subgraph(nodes, edges):
    """Convert graph structure to readable text for the LLM."""
    lines = ["Knowledge Graph Context:"]
    lines.append(f"Entities: {', '.join(sorted(nodes)[:30])}")
    lines.append("\nRelationships:")
    
    # Group by subject for readability
    by_subject = {}
    for s, p, o in edges:
        if s not in by_subject:
            by_subject[s] = []
        by_subject[s].append((p, o))
    
    for subject, relations in sorted(by_subject.items()):
        rel_strs = [f"{p} -> {o}" for p, o in relations[:5]]  # Limit per node
        lines.append(f"  {subject}: {' | '.join(rel_strs)}")
    
    return "\n".join(lines)

def graphrag_query(question, max_hops=2, top_k_nodes=3):
    """Answer a question using GraphRAG (entity extraction + BFS + LLM)."""
    # Step 1: Extract entities from question
    entities = extract_entities_from_question(question)
    
    # Step 2: Find seed nodes in graph
    seed_nodes = find_seed_nodes(entities, G, top_k=top_k_nodes)
    
    # Step 3: BFS traversal
    if not seed_nodes:
        # Fall back to using entities as direct queries
        seed_nodes = [e for e in entities if e in G.nodes()][:top_k_nodes]
    
    subgraph_nodes, subgraph_edges = bfs_subgraph(G, seed_nodes, max_hops=max_hops)
    
    # Step 4: Textualize subgraph
    graph_context = textualize_subgraph(subgraph_nodes, subgraph_edges)
    
    # Step 5: Generate answer
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful assistant with access to a knowledge graph about AI companies. Answer the question using the graph relationships provided. Be specific and cite the relationships you're using."},
            {"role": "user", "content": f"{graph_context}\n\nQuestion: {question}"}
        ],
        temperature=0
    )
    
    answer = response.choices[0].message.content
    return {
        "answer": answer,
        "seed_nodes": seed_nodes,
        "entities_found": entities,
        "subgraph_size": len(subgraph_nodes),
        "num_edges": len(subgraph_edges)
    }

# Quick test
print("Testing GraphRAG...")
test_result = graphrag_query("Who founded Anthropic?")
print(f"Q: Who founded Anthropic?")
print(f"Seed nodes: {test_result['seed_nodes']}")
print(f"Subgraph size: {test_result['subgraph_size']} nodes, {test_result['num_edges']} edges")
print(f"A: {test_result['answer'][:300]}...")

## Cell 10: Benchmark — 5 Multi-Hop Questions

We test both systems on 5 challenging multi-hop questions that require connecting multiple entities.

Multi-hop questions require reasoning like:
- Q: "Which Anthropic founders worked at OpenAI?"
- Chain: Anthropic → founder_of ← Dario Amodei → previously_worked_at → OpenAI

In [ ]:
%%time

benchmark_questions = [
    {
        "question": "Which AI companies were co-founded by people who previously worked at Google?",
        "expected_entities": ["Google", "Anthropic", "DeepMind"],
        "type": "multi-hop"
    },
    {
        "question": "What companies did Sam Altman co-found or lead?",
        "expected_entities": ["Sam Altman", "OpenAI"],
        "type": "multi-hop"
    },
    {
        "question": "Which AI companies have received investment from major tech corporations?",
        "expected_entities": ["Microsoft", "Google", "Amazon"],
        "type": "multi-hop"
    },
    {
        "question": "What is the relationship between Elon Musk and AI companies?",
        "expected_entities": ["Elon Musk", "OpenAI", "xAI"],
        "type": "multi-hop"
    },
    {
        "question": "Which founders of Anthropic previously worked at OpenAI?",
        "expected_entities": ["Anthropic", "OpenAI", "Dario Amodei"],
        "type": "multi-hop"
    }
]

results = []

for i, bq in enumerate(benchmark_questions):
    q = bq["question"]
    print(f"\n{'='*60}")
    print(f"Question {i+1}: {q}")
    print(f"{'='*60}")
    
    # --- Flat RAG ---
    print("Running Flat RAG...")
    t0 = time.time()
    try:
        flat_answer = flat_rag_query(q)
    except Exception as e:
        flat_answer = f"Error: {e}"
    flat_latency = time.time() - t0
    print(f"  Latency: {flat_latency:.2f}s")
    print(f"  Answer preview: {flat_answer[:150]}...")
    
    time.sleep(0.5)
    
    # --- GraphRAG ---
    print("Running GraphRAG...")
    t0 = time.time()
    try:
        graph_result = graphrag_query(q)
        graph_answer = graph_result["answer"]
        graph_meta = {
            "seed_nodes": graph_result["seed_nodes"],
            "subgraph_size": graph_result["subgraph_size"]
        }
    except Exception as e:
        graph_answer = f"Error: {e}"
        graph_meta = {"seed_nodes": [], "subgraph_size": 0}
    graph_latency = time.time() - t0
    print(f"  Latency: {graph_latency:.2f}s")
    print(f"  Seed nodes: {graph_meta['seed_nodes']}")
    print(f"  Subgraph size: {graph_meta['subgraph_size']} nodes")
    print(f"  Answer preview: {graph_answer[:150]}...")
    
    results.append({
        "question": q,
        "expected_entities": bq["expected_entities"],
        "flat_answer": flat_answer,
        "graph_answer": graph_answer,
        "flat_latency": flat_latency,
        "graph_latency": graph_latency,
        "graph_meta": graph_meta
    })
    
    time.sleep(1)  # Rate limiting between questions

print(f"\nBenchmark complete: {len(results)} questions answered")

## Cell 11: Evaluation & Results Table

We use an LLM judge to score each answer on a 0-1 scale:
- Does the answer correctly address the question?
- Are the expected entities mentioned?

Then we build a summary DataFrame and compare approaches.

In [ ]:
def llm_judge(question, answer, expected_entities):
    """Score an answer using LLM as a judge (0.0 to 1.0)."""
    try:
        expected_str = ", ".join(expected_entities)
        prompt = f"""You are an evaluator for question-answering systems.

Question: {question}

Answer: {answer}

Expected key entities that should be mentioned: {expected_str}

Score this answer from 0.0 to 1.0 based on:
1. Correctness and relevance (0-0.5): Does it correctly address the question?
2. Entity coverage (0-0.3): How many expected entities are mentioned?
3. Specificity (0-0.2): Does it give specific facts rather than vague statements?

Return JSON: {{"score": <float 0-1>, "reasoning": "<brief explanation>"}}"""  
        
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"},
            temperature=0
        )
        result = json.loads(response.choices[0].message.content)
        return float(result.get("score", 0.0)), result.get("reasoning", "")
    except Exception as e:
        print(f"Judge error: {e}")
        return 0.0, str(e)

# Evaluate all results
print("Running LLM evaluation...")
eval_data = []

for i, result in enumerate(results):
    q_short = result["question"][:60] + "..." if len(result["question"]) > 60 else result["question"]
    print(f"\nEvaluating Q{i+1}: {q_short}")
    
    # Score Flat RAG
    flat_score, flat_reasoning = llm_judge(
        result["question"], result["flat_answer"], result["expected_entities"]
    )
    print(f"  Flat RAG score: {flat_score:.2f} — {flat_reasoning[:80]}")
    time.sleep(0.3)
    
    # Score GraphRAG
    graph_score, graph_reasoning = llm_judge(
        result["question"], result["graph_answer"], result["expected_entities"]
    )
    print(f"  GraphRAG score: {graph_score:.2f} — {graph_reasoning[:80]}")
    time.sleep(0.3)
    
    eval_data.append({
        "Question": result["question"][:80] + "..." if len(result["question"]) > 80 else result["question"],
        "FlatRAG_Score": round(flat_score, 2),
        "GraphRAG_Score": round(graph_score, 2),
        "FlatRAG_Latency": round(result["flat_latency"], 2),
        "GraphRAG_Latency": round(result["graph_latency"], 2),
        "FlatRAG_Answer": result["flat_answer"][:150] + "...",
        "GraphRAG_Answer": result["graph_answer"][:150] + "..."
    })

# Create results DataFrame
df_results = pd.DataFrame(eval_data)

print("\n" + "="*80)
print("BENCHMARK RESULTS")
print("="*80)
display(df_results[["Question", "FlatRAG_Score", "GraphRAG_Score", "FlatRAG_Latency", "GraphRAG_Latency"]])

# Summary statistics
avg_flat_score = df_results["FlatRAG_Score"].mean()
avg_graph_score = df_results["GraphRAG_Score"].mean()
avg_flat_latency = df_results["FlatRAG_Latency"].mean()
avg_graph_latency = df_results["GraphRAG_Latency"].mean()

print(f"\n{'='*50}")
print("SUMMARY STATISTICS")
print(f"{'='*50}")
print(f"{'Metric':<30} {'Flat RAG':>12} {'GraphRAG':>12}")
print(f"{'-'*54}")
print(f"{'Avg Score':<30} {avg_flat_score:>12.3f} {avg_graph_score:>12.3f}")
print(f"{'Avg Latency (s)':<30} {avg_flat_latency:>12.2f} {avg_graph_latency:>12.2f}")
print(f"{'Min Score':<30} {df_results['FlatRAG_Score'].min():>12.2f} {df_results['GraphRAG_Score'].min():>12.2f}")
print(f"{'Max Score':<30} {df_results['FlatRAG_Score'].max():>12.2f} {df_results['GraphRAG_Score'].max():>12.2f}")

if avg_flat_score > 0:
    improvement = ((avg_graph_score - avg_flat_score) / avg_flat_score) * 100
    print(f"\nGraphRAG improvement: {improvement:+.1f}%")
else:
    print("\nGraphRAG improvement: N/A (Flat RAG score is 0)")

In [ ]:
# Visualization: Score comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Score comparison bar chart
x = range(len(benchmark_questions))
width = 0.35
ax1 = axes[0]
bars1 = ax1.bar([xi - width/2 for xi in x], df_results["FlatRAG_Score"], width, label="Flat RAG", color="#4a90d9", alpha=0.8)
bars2 = ax1.bar([xi + width/2 for xi in x], df_results["GraphRAG_Score"], width, label="GraphRAG", color="#e8a838", alpha=0.8)
ax1.set_xlabel("Question #")
ax1.set_ylabel("LLM Judge Score (0-1)")
ax1.set_title("Answer Quality: Flat RAG vs GraphRAG", fontweight="bold")
ax1.set_xticks(list(x))
ax1.set_xticklabels([f"Q{i+1}" for i in x])
ax1.set_ylim(0, 1.1)
ax1.legend()
ax1.axhline(y=avg_flat_score, color="#4a90d9", linestyle="--", alpha=0.5, label=f"Flat avg: {avg_flat_score:.2f}")
ax1.axhline(y=avg_graph_score, color="#e8a838", linestyle="--", alpha=0.5, label=f"Graph avg: {avg_graph_score:.2f}")

# Latency comparison
ax2 = axes[1]
bars3 = ax2.bar([xi - width/2 for xi in x], df_results["FlatRAG_Latency"], width, label="Flat RAG", color="#4a90d9", alpha=0.8)
bars4 = ax2.bar([xi + width/2 for xi in x], df_results["GraphRAG_Latency"], width, label="GraphRAG", color="#e8a838", alpha=0.8)
ax2.set_xlabel("Question #")
ax2.set_ylabel("Latency (seconds)")
ax2.set_title("Response Latency: Flat RAG vs GraphRAG", fontweight="bold")
ax2.set_xticks(list(x))
ax2.set_xticklabels([f"Q{i+1}" for i in x])
ax2.legend()

plt.tight_layout()
plt.savefig("/Users/jin/Day19/benchmark_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved to benchmark_results.png")

## Cell 12: Cost Analysis

Approximate token usage and cost breakdown for the entire pipeline.

OpenAI pricing (approximate as of early 2025):
- `gpt-4o-mini`: $0.15/1M input tokens, $0.60/1M output tokens
- `text-embedding-3-small`: $0.02/1M tokens

In [ ]:
# Cost Analysis
# Pricing (USD per 1M tokens, as of early 2025)
GPT4O_MINI_INPUT_PRICE = 0.15 / 1_000_000   # per token
GPT4O_MINI_OUTPUT_PRICE = 0.60 / 1_000_000  # per token
EMBED_PRICE = 0.02 / 1_000_000               # per token

# Estimate token counts
num_articles = len(articles)
avg_article_chars = sum(len(a["content"]) for a in articles) / max(num_articles, 1)
avg_article_tokens = int(avg_article_chars / 4)  # ~4 chars per token

# 1. Entity extraction
extraction_input_tokens = num_articles * (avg_article_tokens + 200)  # article + system prompt
extraction_output_tokens = num_articles * 300  # ~20 triples * 15 tokens each
extraction_cost = (extraction_input_tokens * GPT4O_MINI_INPUT_PRICE + 
                   extraction_output_tokens * GPT4O_MINI_OUTPUT_PRICE)

# 2. Node embeddings
num_nodes = G.number_of_nodes()
avg_node_chars = sum(len(n) for n in G.nodes()) / max(num_nodes, 1)
node_embed_tokens = int(num_nodes * avg_node_chars / 4)
node_embed_cost = node_embed_tokens * EMBED_PRICE

# 3. Chunk embeddings
chunk_embed_tokens = int(sum(len(c) for c in all_chunks) / 4)
chunk_embed_cost = chunk_embed_tokens * EMBED_PRICE

# 4. Query processing (benchmark + entity extraction)
num_questions = len(benchmark_questions)
# Per question: entity extraction (2 calls) + flat RAG (1 call) + graphRAG (1 call) + judge (2 calls)
query_input_tokens = num_questions * (200 + 500 + 1500 + 1500 + 400)  # sum of all calls per question
query_output_tokens = num_questions * (50 + 200 + 200 + 200 + 100)    # outputs
query_cost = (query_input_tokens * GPT4O_MINI_INPUT_PRICE + 
              query_output_tokens * GPT4O_MINI_OUTPUT_PRICE)

total_cost = extraction_cost + node_embed_cost + chunk_embed_cost + query_cost

print("="*65)
print("COST ANALYSIS SUMMARY")
print("="*65)
print(f"{'Stage':<35} {'Tokens':>10} {'Cost (USD)':>12}")
print("-"*65)
print(f"{'Entity Extraction (LLM)':<35} {extraction_input_tokens+extraction_output_tokens:>10,} {'$'+f'{extraction_cost:.4f}':>12}")
print(f"{'Node Embeddings':<35} {node_embed_tokens:>10,} {'$'+f'{node_embed_cost:.4f}':>12}")
print(f"{'Chunk Embeddings (Flat RAG)':<35} {chunk_embed_tokens:>10,} {'$'+f'{chunk_embed_cost:.4f}':>12}")
print(f"{'Query Processing (Benchmark)':<35} {query_input_tokens+query_output_tokens:>10,} {'$'+f'{query_cost:.4f}':>12}")
print("-"*65)
print(f"{'TOTAL':<35} {extraction_input_tokens+extraction_output_tokens+node_embed_tokens+chunk_embed_tokens+query_input_tokens+query_output_tokens:>10,} {'$'+f'{total_cost:.4f}':>12}")
print("="*65)
print(f"\nKey stats:")
print(f"  Articles processed: {num_articles}")
print(f"  Total triples: {len(all_triples)}")
print(f"  Graph nodes: {num_nodes}")
print(f"  Graph edges: {G.number_of_edges()}")
print(f"  Text chunks: {len(all_chunks)}")
print(f"  Benchmark questions: {num_questions}")
print(f"\nNote: Actual costs may differ based on exact token counts.")
print(f"Prices based on GPT-4o-mini ($0.15/$0.60 per 1M in/out) and")
print(f"text-embedding-3-small ($0.02 per 1M tokens).")

## Cell 13: Conclusions

Summary and recommendations for when to use each approach.

In [ ]:
# Compute final summary metrics
avg_flat = df_results["FlatRAG_Score"].mean() if len(df_results) > 0 else 0
avg_graph = df_results["GraphRAG_Score"].mean() if len(df_results) > 0 else 0
avg_flat_lat = df_results["FlatRAG_Latency"].mean() if len(df_results) > 0 else 0
avg_graph_lat = df_results["GraphRAG_Latency"].mean() if len(df_results) > 0 else 0
improvement = ((avg_graph - avg_flat) / avg_flat * 100) if avg_flat > 0 else 0
lat_overhead = ((avg_graph_lat - avg_flat_lat) / avg_flat_lat * 100) if avg_flat_lat > 0 else 0

summary = f"""
╔══════════════════════════════════════════════════════════════════╗
║          GRAPHRAG vs FLAT RAG — FINAL COMPARISON                 ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  ACCURACY (LLM Judge, 0-1 scale)                                 ║
║  ─────────────────────────────────────────────────────────────── ║
║  Flat RAG avg score:   {avg_flat:.3f}                              ║
║  GraphRAG avg score:   {avg_graph:.3f}                              ║
║  GraphRAG improvement: {improvement:+.1f}%                            ║
║                                                                  ║
║  LATENCY                                                         ║
║  ─────────────────────────────────────────────────────────────── ║
║  Flat RAG avg:         {avg_flat_lat:.2f}s                             ║
║  GraphRAG avg:         {avg_graph_lat:.2f}s                             ║
║  Latency overhead:     {lat_overhead:+.1f}%                            ║
║                                                                  ║
║  ESTIMATED COST                                                  ║
║  ─────────────────────────────────────────────────────────────── ║
║  Total pipeline cost:  ${total_cost:.4f} USD                      ║
║  Cost per question:    ${total_cost/max(num_questions,1):.4f} USD                      ║
║                                                                  ║
╠══════════════════════════════════════════════════════════════════╣
║  WHEN TO USE EACH APPROACH                                       ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  USE FLAT RAG WHEN:                                              ║
║  • Questions are factual and self-contained                       ║
║  • Low latency is critical                                        ║
║  • Knowledge domain lacks rich entity relationships               ║
║  • Budget is tight (lower per-query cost)                         ║
║  • Documents are unstructured prose                               ║
║                                                                  ║
║  USE GRAPHRAG WHEN:                                              ║
║  • Questions require multi-hop reasoning                          ║
║  • Relationships between entities are key                         ║
║  • Domain has rich structured knowledge (org charts, etc.)        ║
║  • Answer quality matters more than latency                       ║
║  • Need explainable reasoning paths                               ║
║                                                                  ║
║  HYBRID APPROACH:                                                ║
║  Consider using GraphRAG for entity-heavy questions and           ║
║  falling back to Flat RAG when no entities are found.             ║
╚══════════════════════════════════════════════════════════════════╝
"""

print(summary)

In [ ]:
# Final comparison visualization
fig, ax = plt.subplots(figsize=(8, 5))

metrics = ["Avg Score\n(0-1)", "Normalized\nLatency"]
# Normalize latency to 0-1 scale for comparison
max_lat = max(avg_flat_lat, avg_graph_lat, 1)
flat_vals = [avg_flat, avg_flat_lat / max_lat]
graph_vals = [avg_graph, avg_graph_lat / max_lat]

x = np.arange(len(metrics))
width = 0.3

ax.bar(x - width/2, flat_vals, width, label="Flat RAG", color="#4a90d9", alpha=0.85)
ax.bar(x + width/2, graph_vals, width, label="GraphRAG", color="#e8a838", alpha=0.85)

# Annotations
for i, (fv, gv) in enumerate(zip(flat_vals, graph_vals)):
    ax.text(i - width/2, fv + 0.01, f"{fv:.2f}", ha="center", va="bottom", fontsize=10)
    ax.text(i + width/2, gv + 0.01, f"{gv:.2f}", ha="center", va="bottom", fontsize=10)

ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=11)
ax.set_ylim(0, 1.2)
ax.set_ylabel("Score / Normalized Value", fontsize=11)
ax.set_title("GraphRAG vs Flat RAG: Quality vs Speed Tradeoff", fontsize=13, fontweight="bold")
ax.legend(fontsize=11)
ax.text(0.98, 0.95, f"GraphRAG accuracy improvement: {improvement:+.1f}%",
        transform=ax.transAxes, ha="right", va="top",
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5), fontsize=10)

plt.tight_layout()
plt.savefig("/Users/jin/Day19/final_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Final comparison chart saved to final_comparison.png")
print("\nNotebook execution complete!")